# Slides / Report   
---

---
# Handson Verilog – Seven Segment Display demo


## FPGA implementation

### Steps
- Environment Check-Up
- Device Setup
- Module 
- Testbench (no need, as this demo direct implementation)
- Constraints 
- Create Vivado Project
- Run RTL Simulation (no need, as this demo direct implementation onl_modified_von_1My)

In [1]:
# Environment Check-Up
! vivado -version
%load_ext vivado_magic

vivado v2025.1 (64-bit)
Tool Version Limit: 2025.05
SW Build 6140274 on Wed May 21 22:58:25 MDT 2025
IP Build 6138677 on Thu May 22 03:10:11 MDT 2025
SharedData Build 6139179 on Tue May 20 17:58:58 MDT 2025
Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.


In [2]:
device_part  = "xc7a35tcpg236-1" # Do not touch
project_name = "A7E_lfsr_nl_modified_von_1M"
design_top   = "lfsr_nl_modified_von_1M"
sim_top      = "lfsr_nl_modified_von_1M_tb"
%vivado prj_clean

[INFO]: build directory does not exist: /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_lfsr_nl_modified_von_1M


## Module and Testbench

# Non-Linearise LFSR

In [1]:
%%vivado_design lfsr_nl_modified_von_1M
module lfsr_nl_modified_von_1M(
    input         sysclk,
    input  [1:0]  SW,
    input  [1:0]  btn,
    output [5:0]  HEX,
    output        DP,
    output [6:0]  SEG,
    output [1:0]  led,
    output        tx
);

    wire rstn   = ~SW[1];
    wire en      =  SW[0];
    wire freeze =  btn[0];

    // --------------------------------------------------------
    // 1. XADC — internal temperature sensor entropy source
    // --------------------------------------------------------
    wire [15:0] xadc_do;
    wire        xadc_drdy;
    wire        xadc_eoc;

    XADC #(
        .INIT_40(16'h9000), .INIT_41(16'h2ef0), .INIT_42(16'h0400),
        .INIT_48(16'h0800), .INIT_49(16'h0000)
    ) xadc_inst (
        .DCLK   (sysclk),
        .DRDY   (xadc_drdy),
        .DO      (xadc_do),
        .EOC    (xadc_eoc),
        .DEN    (xadc_eoc),
        .DADDR  (7'h00),
        .RESET  (~rstn),
        .VP(1'b0), .VN(1'b0), .VAUXP(16'h0), .VAUXN(16'h0),
        .CONVST(1'b0), .CONVSTCLK(1'b0), .DI(16'h0), .DWE(1'b0)
    );

    // --------------------------------------------------------
    // 2. von_1M Neumann Corrector Logic (Fixes Monobit Fail)
    // --------------------------------------------------------
    reg vn_state, vn_bit_a, vn_valid, clean_bit;

    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) begin
            vn_state <= 1'b0;
            vn_valid <= 1'b0;
        end else if (xadc_drdy) begin
            if (!vn_state) begin
                vn_bit_a <= xadc_do[0]; 
                vn_state <= 1'b1;
                vn_valid <= 1'b0;
            end else begin
                vn_state <= 1'b0;
                if (vn_bit_a != xadc_do[0]) begin
                    clean_bit <= vn_bit_a;
                    vn_valid  <= 1'b1;
                end else vn_valid <= 1'b0;
            end
        end else vn_valid <= 1'b0;
    end

    // --------------------------------------------------------
    // 3. 16-bit LFSR & Free-running Counter
    // --------------------------------------------------------
    reg [15:0] lfsr;
    wire lfsr_feedback = lfsr[15] ^ lfsr[14] ^ lfsr[12] ^ lfsr[3] ^ clean_bit;

    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) lfsr <= 16'hACE1;
        else if (vn_valid) lfsr <= {lfsr[14:0], lfsr_feedback} + 16'h9E37;
    end

    // ADDED: Missing free_ctr definition
    reg [15:0] free_ctr;
    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) free_ctr <= 16'hFFFF;
        else       free_ctr <= free_ctr + 16'd31957; 
    end

    // --------------------------------------------------------
    // 4. Whitening Functions (xor_mix & S-box)
    // --------------------------------------------------------
    function [15:0] xor_mix;
        input [15:0] x;
        reg   [15:0] h;
        begin
            h = x ^ (x >> 7);
            h = h * 16'h9E37;
            h = h ^ (h >> 9);
            h = h * 16'h8445; 
            xor_mix = h ^ (h >> 13);
        end
    endfunction

    function [3:0] sbox;
        input [3:0] x;
        case (x)
            4'h0: sbox = 4'hE; 4'h1: sbox = 4'h4; 4'h2: sbox = 4'hD; 4'h3: sbox = 4'h1;
            4'h4: sbox = 4'h2; 4'h5: sbox = 4'hF; 4'h6: sbox = 4'hB; 4'h7: sbox = 4'h8;
            4'h8: sbox = 4'h3; 4'h9: sbox = 4'hA; 4'hA: sbox = 4'h6; 4'hB: sbox = 4'hC;
            4'hC: sbox = 4'h5; 4'hD: sbox = 4'h9; 4'hE: sbox = 4'h0; 4'hF: sbox = 4'h7;
        endcase
    endfunction

    function [15:0] sbox_16;
        input [15:0] x;
        sbox_16 = {sbox(x[15:12]), sbox(x[11:8]), sbox(x[7:4]), sbox(x[3:0])};
    endfunction

    // --------------------------------------------------------
    // 5. Display & Update Logic
    // --------------------------------------------------------
    reg [23:0] update_ctr;
    reg        update_tick;
    reg [15:0] display_word;
    //--------Change needed-------
    // Change "24'd999999" to 24'd....
    // -----------------------------
    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) begin
            update_ctr  <= 24'd0;
            update_tick <= 1'b0;
        end else if (update_ctr >= 24'd999999) begin    
            update_ctr  <= {8'd0, free_ctr[7:0], lfsr[7:0]};
            update_tick <= 1'b1;
        end else begin
            update_tick <= 1'b0;
            update_ctr <= update_ctr + 1;
        end
    end

    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) display_word <= 16'hACE1;
        else if (update_tick & ~freeze)
            display_word <= xor_mix((sbox_16(lfsr) + sbox_16(free_ctr)));
    end

    wire [3:0] digit3 = display_word[15:12];
    wire [3:0] digit2 = display_word[11:8];
    wire [3:0] digit1 = display_word[7:4];l
    wire [3:0] digit0 = display_word[3:0];

    // --------------------------------------------------------
    // 7-seg mux (~1 kHz refresh)
    // --------------------------------------------------------
    reg [13:0] mux_ctr;
    reg [1:0]  digit_sel;

    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) begin
            mux_ctr   <= 14'd0;
            digit_sel <= 2'd0;
        end else if (mux_ctr == 14'd11999) begin
            mux_ctr   <= 14'd0;
            digit_sel <= digit_sel + 1;
        end else begin
            mux_ctr <= mux_ctr + 1;
        end
    end

    function [6:0] hex2seg;
        input [3:0] d;
        case (d)
            4'h0: hex2seg = 7'b1000000;
            4'h1: hex2seg = 7'b1111001;
            4'h2: hex2seg = 7'b0100100;
            4'h3: hex2seg = 7'b0110000;
            4'h4: hex2seg = 7'b0011001;
            4'h5: hex2seg = 7'b0010010;
            4'h6: hex2seg = 7'b0000010;
            4'h7: hex2seg = 7'b1111000;
            4'h8: hex2seg = 7'b0000000;
            4'h9: hex2seg = 7'b0010000;
            4'hA: hex2seg = 7'b0001000;
            4'hB: hex2seg = 7'b0000011;
            4'hC: hex2seg = 7'b1000110;
            4'hD: hex2seg = 7'b0100001;
            4'hE: hex2seg = 7'b0000110;
            4'hF: hex2seg = 7'b0001110;
        endcase
    endfunction

    reg [3:0] cur_nibble;
    always @(*) begin
        case (digit_sel)
            2'd0: cur_nibble = digit3;
            2'd1: cur_nibble = digit2;
            2'd2: cur_nibble = digit1;
            2'd3: cur_nibble = digit0;
        endcase
    end

    assign SEG    = ~hex2seg(cur_nibble);
    assign HEX[3] = ~(digit_sel == 2'd0);
    assign HEX[2] = ~(digit_sel == 2'd1);
    assign HEX[1] = ~(digit_sel == 2'd2);
    assign HEX[0] = ~(digit_sel == 2'd3);
    assign HEX[5:4] = 2'b11;
    assign DP = 1'b0;

    // --------------------------------------------------------
    // LEDs
    // CHANGED: led[0] and led[1] now show raw XADC entropy bits
    // If XADC is working → both LEDs flicker irregularly
    // If XADC is dead   → both LEDs stay solid or blink regularly
    // Comment these out and restore originals after verification
    // --------------------------------------------------------
    
    // Original LED behavior (restore after XADC verified):
    reg [23:0] hb_ctr;
    reg        led0_reg, led1_reg;
    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) begin
            hb_ctr   <= 24'd0;
            led0_reg <= 1'b0;
            led1_reg <= 1'b0;
        end else begin
            hb_ctr   <= hb_ctr + 1;
            led0_reg <= en;
            led1_reg <= hb_ctr[23];
        end
    end
    assign led[0] = led0_reg;
    assign led[1] = led1_reg;

    // --------------------------------------------------------
    // UART TX — 115200 baud
    // Sends "XXXX\r\n" every update_tick
    // --------------------------------------------------------
    localparam CLKS_PER_BIT = 104;

    function [7:0] nibble2ascii;
        input [3:0] n;
        nibble2ascii = (n < 4'd10) ? (8'd48 + n) : (8'd55 + n);
    endfunction

    reg [7:0]  tx_buf [0:5];
    reg [2:0]  tx_byte_idx;
    reg [3:0]  tx_bit_idx;
    reg [6:0]  tx_clk_ctr;
    reg        tx_busy;
    reg        tx_reg;

    assign tx = tx_reg;

    always @(posedge sysclk or negedge rstn) begin
        if (!rstn) begin
            tx_busy     <= 1'b0;
            tx_reg      <= 1'b1;
            tx_byte_idx <= 3'd0;
            tx_bit_idx  <= 4'd0;
            tx_clk_ctr  <= 7'd0;
        end else begin

            if (update_tick && !tx_busy) begin
                tx_buf[0] <= nibble2ascii(display_word[15:12]);
                tx_buf[1] <= nibble2ascii(display_word[11:8]);
                tx_buf[2] <= nibble2ascii(display_word[7:4]);
                tx_buf[3] <= nibble2ascii(display_word[3:0]);
                tx_buf[4] <= 8'h0D; // CR
                tx_buf[5] <= 8'h0A; // LF
                tx_busy     <= 1'b1;
                tx_byte_idx <= 3'd0;
                tx_bit_idx  <= 4'd0;
                tx_clk_ctr  <= 7'd0;
            end

            if (tx_busy) begin
                if (tx_clk_ctr < CLKS_PER_BIT - 1) begin
                    tx_clk_ctr <= tx_clk_ctr + 1;
                end else begin
                    tx_clk_ctr <= 7'd0;
                    case (tx_bit_idx)
                        4'd0: tx_reg <= 1'b0;
                        4'd1: tx_reg <= tx_buf[tx_byte_idx][0];
                        4'd2: tx_reg <= tx_buf[tx_byte_idx][1];
                        4'd3: tx_reg <= tx_buf[tx_byte_idx][2];
                        4'd4: tx_reg <= tx_buf[tx_byte_idx][3];
                        4'd5: tx_reg <= tx_buf[tx_byte_idx][4];
                        4'd6: tx_reg <= tx_buf[tx_byte_idx][5];
                        4'd7: tx_reg <= tx_buf[tx_byte_idx][6];
                        4'd8: tx_reg <= tx_buf[tx_byte_idx][7];
                        4'd9: begin
                            tx_reg <= 1'b1;
                            if (tx_byte_idx == 3'd5) begin
                                tx_busy <= 1'b0;
                            end else begin
                                tx_byte_idx <= tx_byte_idx + 1;
                                tx_bit_idx  <= 4'd0;
                            end
                        end
                    endcase
                    if (tx_bit_idx < 4'd9) tx_bit_idx <= tx_bit_idx + 1;
                end
            end

        end
    end

endmodule

UsageError: Cell magic `%%vivado_design` not found.


---

In [4]:
%%vivado_xdc A7E_lfsr_nl_modified_von_1M

#### On CMOD A7 35T Board ########################################################
## 12 MHz Clock
set_property -dict { PACKAGE_PIN L17   IOSTANDARD LVCMOS33 } [get_ports { sysclk }]; #IO_L12P_T1_MRCC_14 Sch=gclk
create_clock -add -name sys_clk_pin -period 83.33 -waveform {0 41.66} [get_ports {sysclk}];

## Onboard LEDs
set_property -dict { PACKAGE_PIN A17   IOSTANDARD LVCMOS33 } [get_ports { led[0] }]; #IO_L12N_T1_MRCC_16 Sch=led[1]
set_property -dict { PACKAGE_PIN C16   IOSTANDARD LVCMOS33 } [get_ports { led[1] }]; #IO_L13P_T2_MRCC_16 Sch=led[2]

## Onboard Buttons
set_property -dict { PACKAGE_PIN A18   IOSTANDARD LVCMOS33 } [get_ports { btn[0] }]; #IO_L19N_T3_VREF_16 Sch=btn[0]
set_property -dict { PACKAGE_PIN B18   IOSTANDARD LVCMOS33 } [get_ports { btn[1] }]; #IO_L19P_T3_16 Sch=btn[1]

#### On Extension Board ##########################################################
### HEX[5]
set_property -dict { PACKAGE_PIN N2    IOSTANDARD LVCMOS33 } [get_ports { HEX[5] }]; #IO_L10P_T1_AD15P_35 Sch=pio[22]
### HEX[4]
set_property -dict { PACKAGE_PIN N1    IOSTANDARD LVCMOS33 } [get_ports { HEX[4] }]; #IO_L10N_T1_AD15N_35 Sch=pio[21]
### HEX[3]
set_property -dict { PACKAGE_PIN L2    IOSTANDARD LVCMOS33 } [get_ports { HEX[3] }]; #IO_L5N_T0_AD13N_35 Sch=pio[14]
### HEX[2]
set_property -dict { PACKAGE_PIN L1    IOSTANDARD LVCMOS33 } [get_ports { HEX[2] }]; #IO_L6N_T0_VREF_35 Sch=pio[13]
### HEX[1]
set_property -dict { PACKAGE_PIN A15   IOSTANDARD LVCMOS33 } [get_ports { HEX[1] }]; #IO_L6N_T0_VREF_16 Sch=pio[07]
### HEX[0]
set_property -dict { PACKAGE_PIN C15   IOSTANDARD LVCMOS33 } [get_ports { HEX[0] }]; #IO_L11P_T1_SRCC_16 Sch=pio[05]

### DP
set_property -dict { PACKAGE_PIN B15   IOSTANDARD LVCMOS33 } [get_ports { DP }];     #IO_L11N_T1_SRCC_16 Sch=pio[08]
### SEG G
set_property -dict { PACKAGE_PIN K3    IOSTANDARD LVCMOS33 } [get_ports { SEG[6] }]; #IO_L7N_T1_AD6N_35 Sch=pio[04]
### SEG F
set_property -dict { PACKAGE_PIN A14   IOSTANDARD LVCMOS33 } [get_ports { SEG[5] }]; #IO_L6P_T0_16 Sch=pio[09]
### SEG E
set_property -dict { PACKAGE_PIN K2    IOSTANDARD LVCMOS33 } [get_ports { SEG[4] }]; #IO_L5P_T0_AD13P_35 Sch=pio[12]
### SEG D
set_property -dict { PACKAGE_PIN J3    IOSTANDARD LVCMOS33 } [get_ports { SEG[3] }]; #IO_L7P_T1_AD6P_35 Sch=pio[10]
### SEG C
set_property -dict { PACKAGE_PIN H1    IOSTANDARD LVCMOS33 } [get_ports { SEG[2] }]; #IO_L3P_T0_DQS_AD5P_35 Sch=pio[06]
### SEG B
set_property -dict { PACKAGE_PIN A16   IOSTANDARD LVCMOS33 } [get_ports { SEG[1] }]; #IO_L12P_T1_MRCC_16 Sch=pio[03]
### SEG A
set_property -dict { PACKAGE_PIN J1    IOSTANDARD LVCMOS33 } [get_ports { SEG[0] }]; #IO_L3N_T0_DQS_AD5N_35 Sch=pio[11]

### LED[9]
 set_property -dict { PACKAGE_PIN T1    IOSTANDARD LVCMOS33 } [get_ports { LED[9] }]; #IO_L3P_T0_DQS_34 Sch=pio[29]
### LED[8]
 set_property -dict { PACKAGE_PIN U1    IOSTANDARD LVCMOS33 } [get_ports { LED[8] }]; #IO_L3N_T0_DQS_34 Sch=pio[31]
### LED[7]
 set_property -dict { PACKAGE_PIN V2    IOSTANDARD LVCMOS33 } [get_ports { LED[7] }]; #IO_L5P_T0_34 Sch=pio[33]
### LED[6]
 set_property -dict { PACKAGE_PIN von_1M    IOSTANDARD LVCMOS33 } [get_ports { LED[6] }]; #IO_L6P_T0_34 Sch=pio[35]
### LED[5]
 set_property -dict { PACKAGE_PIN von_1M    IOSTANDARD LVCMOS33 } [get_ports { LED[5] }]; #IO_L11N_T1_SRCC_34 Sch=pio[37]
### LED[4]
 set_property -dict { PACKAGE_PIN V5    IOSTANDARD LVCMOS33 } [get_ports { LED[4] }]; #IO_L16N_T2_34 Sch=pio[39]
### LED[3]
 set_property -dict { PACKAGE_PIN U5    IOSTANDARD LVCMOS33 } [get_ports { LED[3] }]; #IO_L16P_T2_34 Sch=pio[41]
### LED[2]
 set_property -dict { PACKAGE_PIN W6    IOSTANDARD LVCMOS33 } [get_ports { LED[2] }]; #IO_L13N_T2_MRCC_34 Sch=pio[43]

### SW[1]
set_property -dict { PACKAGE_PIN W7    IOSTANDARD LVCMOS33 } [get_ports { SW[1] }];  #IO_L13P_T2_MRCC_34 Sch=pio[46]
### SW[0]
set_property -dict { PACKAGE_PIN V8    IOSTANDARD LVCMOS33 } [get_ports { SW[0] }];  #IO_L14N_T2_SRCC_34 Sch=pio[48]

set_property -dict { PACKAGE_PIN J18   IOSTANDARD LVCMOS33 } [get_ports { tx }]; #UART TX
## ============================================================
## Timing exceptions for the ring oscillator
## ============================================================
#set_false_path -from [get_nets {ring[*]}] -to [get_pins ff0_reg/D]
#set_multicycle_path -setup 2 -from [get_pins ff0_reg/Q] -to [get_pins ff1_reg/D]
#set_property DONT_TOUCH TRUE [get_nets {ring[*]}]
#set_property ALLOW_COMBINATORIAL_LOOPS TRUE [get_nets {ring[29]}]

set_property DONT_TOUCH TRUE [get_cells {ring_stages[*].inv}]
set_property DONT_TOUCH TRUE [get_cells {ring_gate}]
set_false_path -from [get_cells {ring_stages[*].inv}] -to [get_cells -hierarchical -filter {NAME =~ *ff0*}]
set_property ALLOW_COMBINATORIAL_LOOPS TRUE [get_nets {ring[29]}]

[INFO]: Creating XDC file: /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_lfsr_nl_modified_von_1M/src/constraints/A7E_lfsr_nl_modified_von_1M.xdc
[INFO]: Vivado project '/home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_lfsr_nl_modified_von_1M/A7E_lfsr_nl_modified_von_1M/A7E_lfsr_nl_modified_von_1M.xpr' not found; skipping auto-add.
[INFO]: The file is saved under build/src and will be picked up when you run '%vivado prj_create'.


## Controls after programming

| Control | Location | Action |
|---|---|---|
| `SW[0]` slide right | Extension board | Starts oscillator — display rolls |
| `SW[1]` slide right | Extension board | Holds reset — display shows `00` |
| `btn[0]` hold | Onboard CMod A7 | Freezes display so you can read the value |
| `led[0]` lit | Onboard CMod A7 | Oscillator is enabled |
| `led[1]` blinking | Onboard CMod A7 | ~0.7 Hz heartbeat — board is alive |

## If digits appear on the wrong 7-seg position
The design drives `HEX[0]` for tens and `HEX[1]` for ones.
If they appear swapped or on the wrong physical digit, adjust
the `HEX` assignments at the bottom of `top_module.v` to match
your extension board's digit ordering.

## Create Vivado Project

In [5]:
%vivado prj_create

[INFO]: Running: vivado -mode batch -source /tmp/tmp95azv14d/cmd.tcl -tclargs A7E_lfsr_nl_modified_von_1M xc7a35tcpg236-1 lfsr_nl_modified_von_1M lfsr_nl_modified_von_1M_tb /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_lfsr_nl_modified_von_1M/src/hdl /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_lfsr_nl_modified_von_1M/src/tb /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_lfsr_nl_modified_von_1M/src/constraints /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_lfsr_nl_modified_von_1M

****** Vivado v2025.1 (64-bit)
  **** SW Build 6140274 on Wed May 21 22:58:25 MDT 2025
  **** IP Build 6138677 on Thu May 22 03:10:11 MDT 2025
  **** SharedData Build 6139179 on Tue May 20 17:58:58 MDT 2025
  **** Start of session at: Thu Apr 16 11:29:02 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

[INFO]: Project name  : A7E_lfsr_nl_modified_von_1M
[INFO]: Device part   : xc7a35tcpg236-1
[INFO]: 

In [6]:
%vivado syn
%vivado imp

[INFO]: Running: vivado -mode batch -source /tmp/tmpmmnk4dbj/cmd.tcl -tclargs A7E_lfsr_nl_modified_von_1M /home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_lfsr_nl_modified_von_1M lfsr_nl_modified_von_1M xc7a35tcpg236-1

****** Vivado v2025.1 (64-bit)
  **** SW Build 6140274 on Wed May 21 22:58:25 MDT 2025
  **** IP Build 6138677 on Thu May 22 03:10:11 MDT 2025
  **** SharedData Build 6139179 on Tue May 20 17:58:58 MDT 2025
  **** Start of session at: Thu Apr 16 11:29:10 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

INFO: [filemgmt 56-3] Default IP Output Path : Could not find the directory '/home/user/ug2605/Dsl_notebook/DSL_Labs/build_A7E_lfsr_nl_modified_von_1M/A7E_lfsr_nl_modified_von_1M/A7E_lfsr_nl_modified_von_1M.gen/sources_1'.
Scanning sources...
Finished scanning sources
[INFO]: Starting synthesis...
[Thu Apr 16 11:29:15 2026] Launched synth_1...
Run output will be captured here: 